In [1]:
#imports

import os
import sys
import json
import tqdm
import pandas as pd
import re

import numpy as np
from scipy import stats
from itertools import combinations
from IPython.display import display

#some important stuff setup

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

results_dir = os.path.join(project_root, "assets", "outputs", "test_results")

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import TEST_DATA_LORE_DESCRIPTION, TEST_DATA_DESING_DOCUMENT, TEST_DATA_SCENARIO
from nir.tests.metrics import compute_world_consistency, compute_interestingness

In [7]:
#models setup

manager = ModelManager()

instruct_small_model_config = ModelConfig(model_name="qwen2.5:3b-instruct", temperature=0.0)
instruct_llm = manager.create_chat_model(name="qwen_3b_evaluation_model_tests", option="ollama", config=instruct_small_model_config)

Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


In [8]:
def recalculate_metrics(pipeline_name, dataset_name, test_data, llm, language="en"):
    json_path = os.path.join(results_dir, pipeline_name, f"{dataset_name}.json")
    if not os.path.exists(json_path):
        print(f"File is not found: {json_path}")
        return
    with open(json_path, "r", encoding="utf-8") as f:
        generated_data = json.load(f)
    tasks = test_data.get("tasks", [])
    base_context = test_data.get("text_summary", "") 

    results = []
    print(f"\nRecalculate metrics for: [ {pipeline_name} ] | [ {dataset_name} ]")
    
    for i, item in enumerate(tqdm.tqdm(generated_data, desc=f"{pipeline_name}")):
        
        gen_text = item.get("generated_text", "")
        gen_text = re.sub(r'</?answer>', '', gen_text)
        category = item.get("category", "default")
        
        query = tasks[i]["query"] if i < len(tasks) else "No query"
        context = base_context 
    
        wc_score = compute_world_consistency(
            original_context=context,
            query=query,
            generated_text=gen_text,
            llm=llm,
            language=language
        )
        
        task_desc = f"Category: {category}\nQuery: {query}"
        int_score = compute_interestingness(
            task_description=task_desc,
            generated_text=gen_text,
            llm=llm,
            language=language
        )
        
        results.append({
            "category": category,
            "query": query,
            "world_consistency": round(wc_score, 4),
            "interestingness": round(int_score, 4)
        })
        
    df = pd.DataFrame(results)
    overall = pd.DataFrame([{
        "category": "OVERALL",
        "query": "-",
        "world_consistency": round(df["world_consistency"].mean(), 4),
        "interestingness": round(df["interestingness"].mean(), 4)
    }])
    final_df = pd.concat([df, overall], ignore_index=True)
    display(final_df)
    return df

In [9]:
pipelines = [
    "Standard RAG", 
    "This NIR (two-staged generation)", 
    "This NIR (one-staged generation)"
]

datasets_map = {
    "lore_description": TEST_DATA_LORE_DESCRIPTION,
    "design_document": TEST_DATA_DESING_DOCUMENT,
    "scenario": TEST_DATA_SCENARIO
}

In [10]:
recalculated_results = {}
print("Start recalculation...\n")

for pipeline in pipelines:
    for ds_name, ds_data in datasets_map.items():
        df_result = recalculate_metrics(pipeline, ds_name, ds_data, instruct_llm, language="en")
        key = f"{pipeline}_{ds_name}"
        recalculated_results[key] = df_result

print("\nRecalculation is finished.")

pipeline_dfs = {pipeline: [] for pipeline in pipelines}

for key, df in recalculated_results.items():
    for pipeline in pipelines:
        if key.startswith(pipeline):
            df_clean = df[df["category"] != "OVERALL"].copy()
            pipeline_dfs[pipeline].append(df_clean)

recalculated_all_pipelines = {}

for pipeline, dfs in pipeline_dfs.items():
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        recalculated_all_pipelines[pipeline] = combined_df
        print(f"Data for '{pipeline}' are combined. Total amount: {len(combined_df)}")

Start recalculation...


Recalculate metrics for: [ Standard RAG ] | [ lore_description ]


Standard RAG: 100%|██████████| 25/25 [04:34<00:00, 10.97s/it]


,category,query,world_consistency,interestingness
0,character description,"Write a short, unstructured character descript...",0.870,0.750
1,character description,Write a structured character description for a...,0.000,0.750
2,character description,Write a short (no more than 15 short sentences...,0.950,0.750
3,character description,Write a short (no more than 15 short sentences...,0.950,0.850
4,character description,Write a short description for an NPC Finger Re...,0.750,0.750
5,location description,Write a short description for a game hub where...,0.780,0.750
6,location description,Write a description for the location of Raya L...,0.850,0.750
7,location description,Write a location description for a manor that ...,0.980,0.850
8,location description,"Write a short description for a region, that l...",0.750,0.700
9,location description,Create a very short (5-7 sentences) descriptio...,0.234,0.850



Recalculate metrics for: [ Standard RAG ] | [ design_document ]


Standard RAG: 100%|██████████| 25/25 [14:35<00:00, 35.04s/it]  


,category,query,world_consistency,interestingness
0,quest,Write a short quest description (12-15 sentenc...,0.0000,0.850
1,quest,Write a multi-step quest (covers three differe...,0.9500,0.850
2,quest,Write a short quest description (10-12 sentenc...,0.0000,0.850
3,quest,Write a short quest description (10-12 sentenc...,0.0000,0.200
4,quest,Write a short quest description (about 20 sent...,0.2500,0.800
5,dialogue,Write a short dialogue scene between Rose Elee...,0.9500,0.850
6,dialogue,Write a short dialogue scene between Thunderbi...,0.8500,0.600
7,dialogue,Write a short dialogue scene between Charlotte...,0.9500,0.850
8,dialogue,Write a short dialogue scene between Cavaricch...,0.9500,0.850
9,dialogue,Write a short dialogue scene between Shablee a...,0.9500,0.850


... (18 outputs left out)


Recalculate metrics for: [ This NIR (one-staged generation) ] | [ scenario ]


This NIR (one-staged generation): 100%|██████████| 25/25 [04:12<00:00, 10.11s/it]


,category,query,world_consistency,interestingness
0,character description,"Create a new character, who is a researcher in...",0.9500,0.600
1,character description,Create a new character from Snezhnaya (he may ...,0.9500,0.200
2,character description,Create a new Fatui character - guardian of the...,0.7500,0.750
3,character description,Create a new female character who connected wi...,0.6500,0.750
4,character description,"Create a new female character, who is an music...",0.8700,0.750
5,item description,Write a short 2-sentence in-game description f...,0.9500,0.200
6,item description,Write a 3-sentenced in-game description for an...,0.9800,0.850
7,item description,Write a 3-sentences in-game description for an...,0.0000,0.850
8,item description,Write a short description in 2-3 sentences for...,0.7500,0.200
9,item description,Write 1-2 sentences for short in-game descript...,0.9500,0.650



Recalculation is finished.
Data for 'Standard RAG' are combined. Total amount: 75
Data for 'This NIR (two-staged generation)' are combined. Total amount: 74
Data for 'This NIR (one-staged generation)' are combined. Total amount: 75


In [11]:
def compute_mean_std_table(pipelines_dict: dict, metrics: list) -> pd.DataFrame:
    rows = []
    for metric in metrics:
        row = {"Metric": metric}
        for name, df in pipelines_dict.items():
            if metric in df.columns:
                vals = df[metric].dropna()
                if len(vals) > 1:
                    mean_val = vals.mean()
                    std_val = vals.std()
                    row[name] = f"{mean_val:.4f} ± {std_val:.4f}"
                else:
                    row[name] = "N/A"
            else:
                row[name] = "N/A"
        rows.append(row)
    return pd.DataFrame(rows)

METRICS_TO_EVALUATE = ['world_consistency', 'interestingness']

print("=" * 80)
print("Mean ± Std for every pipeline")
print("=" * 80)
mean_std_df = compute_mean_std_table(recalculated_all_pipelines, METRICS_TO_EVALUATE)
display(mean_std_df)

def pairwise_comparison_with_delta(
    df_baseline: pd.DataFrame,
    df_proposed: pd.DataFrame,
    metrics: list,
    baseline_name: str,
    proposed_name: str,
    alpha: float = 0.05
) -> pd.DataFrame:

    merged = pd.merge(
        df_baseline, df_proposed, 
        on=['category', 'query'], 
        suffixes=('_base', '_prop')
    )
    
    results = []
    for metric in metrics:
        col_base = f"{metric}_base"
        col_prop = f"{metric}_prop"
        
        if col_base not in merged.columns or col_prop not in merged.columns:
            continue
            
        valid_data = merged[[col_base, col_prop]].dropna()
        if len(valid_data) < 2:
            continue
            
        base_vals = valid_data[col_base].values
        prop_vals = valid_data[col_prop].values
        
        _, p_norm_base = stats.shapiro(base_vals)
        _, p_norm_prop = stats.shapiro(prop_vals)
        is_normal = (p_norm_base > 0.05) and (p_norm_prop > 0.05)
        
        if is_normal:
            t_stat, p_value = stats.ttest_rel(prop_vals, base_vals, alternative='greater')
            test_used = "Paired t-test"
        else:
            w_stat, p_value = stats.wilcoxon(prop_vals, base_vals, alternative='greater')
            test_used = "Wilcoxon signed-rank"
            
        diffs = prop_vals - base_vals
        std_diff = np.std(diffs, ddof=1)
        cohens_d = np.mean(diffs) / std_diff if std_diff > 0 else 0.0

        abs_d = abs(cohens_d)
        if abs_d < 0.2:
            effect_interp = "negligible"
        elif abs_d < 0.5:
            effect_interp = "small"
        elif abs_d < 0.8:
            effect_interp = "medium"
        else:
            effect_interp = "large"
            
        mean_base = np.mean(base_vals)
        mean_prop = np.mean(prop_vals)
        if mean_base != 0:
            delta_pct = ((mean_prop - mean_base) / abs(mean_base)) * 100
        else:
            delta_pct = 0.0
            
        results.append({
            "Metric": metric,
            f"{baseline_name} (Mean)": round(mean_base, 4),
            f"{proposed_name} (Mean)": round(mean_prop, 4),
            "Delta (%)": round(delta_pct, 2),
            "p-value": round(p_value, 4),
            f"Significant (p < {alpha})": "yes" if p_value < alpha else "no",
            "Cohen's d": round(cohens_d, 4),
            "Effect's interpretation": effect_interp,
            "Test used": test_used,
            "N pairs": len(valid_data)
        })
        
    return pd.DataFrame(results)

pipeline_names = list(recalculated_all_pipelines.keys())
pairs = list(combinations(pipeline_names, 2))

print("\n" + "=" * 80)
print(f"Paired comparison (Total amount of pairs: {len(pairs)})")
print("=" * 80)

all_comparisons_summary = []

for base_name, prop_name in pairs:
    print(f"\n{'─' * 80}")
    print(f"Comparison: [ {base_name} ]  vs  [ {prop_name} ]")
    print(f"{'─' * 80}")
    
    res_df = pairwise_comparison_with_delta(
        df_baseline=recalculated_all_pipelines[base_name],
        df_proposed=recalculated_all_pipelines[prop_name],
        metrics=METRICS_TO_EVALUATE,
        baseline_name=base_name,
        proposed_name=prop_name
    )
    
    display(res_df)
    
    for _, row in res_df.iterrows():
        all_comparisons_summary.append({
            "Baseline": base_name,
            "Proposed": prop_name,
            "Metric": row["Metric"],
            "Delta (%)": row["Delta (%)"],
            "p-value": row["p-value"],
            "Significant": row[f"Significant (p < 0.05)"],
            "Cohen's d": row["Cohen's d"],
            "Effect": row["Effect's interpretation"]
        })

print("\n" + "=" * 80)
print("FINAL TABLE")
print("=" * 80)
summary_df = pd.DataFrame(all_comparisons_summary)

summary_df = summary_df.sort_values(by=["Metric", "Baseline", "Proposed"]).reset_index(drop=True)
display(summary_df)

Mean ± Std for every pipeline


,Metric,Standard RAG,This NIR (two-staged generation),This NIR (one-staged generation)
0,world_consistency,0.6677 ± 0.3401,0.6353 ± 0.3808,0.6623 ± 0.3569
1,interestingness,0.7133 ± 0.1891,0.6455 ± 0.2536,0.6867 ± 0.2080



Paired comparison (Total amount of pairs: 3)

────────────────────────────────────────────────────────────────────────────────
Comparison: [ Standard RAG ]  vs  [ This NIR (two-staged generation) ]
────────────────────────────────────────────────────────────────────────────────


,Metric,Standard RAG (Mean),This NIR (two-staged generation) (Mean),Delta (%),p-value,Significant (p < 0.05),Cohen's d,Effect's interpretation,Test used,N pairs
0,world_consistency,0.6798,0.6408,-5.74,0.7937,no,-0.1064,negligible,Wilcoxon signed-rank,73
1,interestingness,0.7110,0.6462,-9.11,0.9831,no,-0.2369,small,Wilcoxon signed-rank,73



────────────────────────────────────────────────────────────────────────────────
Comparison: [ Standard RAG ]  vs  [ This NIR (one-staged generation) ]
────────────────────────────────────────────────────────────────────────────────


,Metric,Standard RAG (Mean),This NIR (one-staged generation) (Mean),Delta (%),p-value,Significant (p < 0.05),Cohen's d,Effect's interpretation,Test used,N pairs
0,world_consistency,0.6767,0.6622,-2.15,0.4909,no,-0.0382,negligible,Wilcoxon signed-rank,74
1,interestingness,0.7128,0.6905,-3.13,0.6604,no,-0.0997,negligible,Wilcoxon signed-rank,74



────────────────────────────────────────────────────────────────────────────────
Comparison: [ This NIR (two-staged generation) ]  vs  [ This NIR (one-staged generation) ]
────────────────────────────────────────────────────────────────────────────────


,Metric,This NIR (two-staged generation) (Mean),This NIR (one-staged generation) (Mean),Delta (%),p-value,Significant (p < 0.05),Cohen's d,Effect's interpretation,Test used,N pairs
0,world_consistency,0.6353,0.6699,5.43,0.2491,no,0.0913,negligible,Wilcoxon signed-rank,74
1,interestingness,0.6455,0.6858,6.24,0.0672,no,0.1471,negligible,Wilcoxon signed-rank,74



FINAL TABLE


,Baseline,Proposed,Metric,Delta (%),p-value,Significant,Cohen's d,Effect
0,Standard RAG,This NIR (one-staged generation),interestingness,-3.13,0.6604,no,-0.0997,negligible
1,Standard RAG,This NIR (two-staged generation),interestingness,-9.11,0.9831,no,-0.2369,small
2,This NIR (two-staged generation),This NIR (one-staged generation),interestingness,6.24,0.0672,no,0.1471,negligible
3,Standard RAG,This NIR (one-staged generation),world_consistency,-2.15,0.4909,no,-0.0382,negligible
4,Standard RAG,This NIR (two-staged generation),world_consistency,-5.74,0.7937,no,-0.1064,negligible
5,This NIR (two-staged generation),This NIR (one-staged generation),world_consistency,5.43,0.2491,no,0.0913,negligible
